# `kitai.retriever` — User Guide

`kitai/retriever.py` provides retrieval strategy helpers for LangChain RAG pipelines.
Each function wraps one LangChain retriever type with consistent input validation
and a clean, raise-on-error contract.

```
kitai/retriever.py
│
├── Vector-store retrieval       (no sparse index, no API key for setup)
│   └── create_retriever()       VectorStore → VectorStoreRetriever
│                                supports 'similarity' | 'mmr' | 'similarity_score_threshold'
│
├── Sparse (keyword) retrieval
│   ├── create_BM25retriever_from_docs()   list[Document] → BM25Retriever
│   └── create_BM25retriever_from_text()   list[str]      → BM25Retriever
│
├── Hybrid retrieval             (sparse + semantic)
│   └── create_hybrid_retriever()  BM25Retriever + VectorStoreRetriever → EnsembleRetriever
│
├── Metadata-filtered retrieval  (requires LLM)
│   └── create_self_query_retriever()  model + VectorStore → SelfQueryRetriever
│
└── Post-retrieval
    └── reorder_docs()           list[Document] → list[Document]  (LongContextReorder)
```

| Retriever | Needs embeddings | Needs LLM | Best for |
|---|---|---|---|
| `create_retriever` (similarity) | yes | no | Semantic similarity |
| `create_retriever` (mmr) | yes | no | Diversity in results |
| `create_BM25retriever_from_*` | no | no | Exact keyword matches |
| `create_hybrid_retriever` | yes | no | Best of both worlds |
| `create_self_query_retriever` | yes | yes | Metadata-filtered queries |
| `reorder_docs` | no | no | Long-context LLM input |

**Prerequisites:**
- Sections 1–7 run with **no API key** — all examples use synthetic data.
- Section 8 (`create_self_query_retriever`) requires an `OPENAI_API_KEY` in your `.env`.

**Related modules:**
- `kitai.index` — FAISS vector store construction and BM25 from docs (canonical source).
- `kitai.query_translation` — query expansion strategies (decompose, step-back, expand).

## Sections
1. [Setup](#1-setup)
2. [Sample corpus](#2-sample-corpus)
3. [`create_retriever` — vector-store retriever](#3-create_retriever)
4. [`create_BM25retriever_from_docs`](#4-create_bm25retriever_from_docs)
5. [`create_BM25retriever_from_text`](#5-create_bm25retriever_from_text)
6. [`reorder_docs` — LongContextReorder](#6-reorder_docs)
7. [`create_hybrid_retriever` — EnsembleRetriever](#7-create_hybrid_retriever)
8. [`create_self_query_retriever`](#8-create_self_query_retriever)
9. [End-to-end pattern](#9-end-to-end-pattern)
10. [Error handling reference](#10-error-handling-reference)

---
## 1 — Setup

In [ ]:
import importlib.util
import logging
import os

import numpy as np
from langchain_core.documents import Document
from langchain_core.embeddings import FakeEmbeddings

from kitai.index import create_vectorstore
from kitai.retriever import (
    create_retriever,
    create_BM25retriever_from_docs,
    create_BM25retriever_from_text,
    create_hybrid_retriever,
    create_self_query_retriever,
    reorder_docs,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

---
## 2 — Sample corpus

All examples share this six-document corpus of finance articles.
Each document carries structured metadata — `topic`, `year`, and `difficulty` — used
in the [self-query section](#8-create_self_query_retriever) to demonstrate metadata filtering.

Every document must have `metadata["id"]` (an integer) so `create_vectorstore` can
key it in the in-memory docstore.

> In production, build this list with `utils.load_text_file` / `chunk_documents` /
> `assign_sequential_ids`.

In [ ]:
DOCS = [
    Document(
        page_content="Short selling profits when a stock price falls below the borrowed price.",
        metadata={"id": 1, "topic": "equities", "year": 2021, "difficulty": "beginner"},
    ),
    Document(
        page_content="Alpha measures excess return generated above a risk-adjusted benchmark.",
        metadata={"id": 2, "topic": "portfolio", "year": 2022, "difficulty": "intermediate"},
    ),
    Document(
        page_content="Maximum drawdown is the largest peak-to-trough decline in portfolio value.",
        metadata={"id": 3, "topic": "risk", "year": 2020, "difficulty": "beginner"},
    ),
    Document(
        page_content="Risk management frameworks define position sizing and stop-loss rules.",
        metadata={"id": 4, "topic": "risk", "year": 2023, "difficulty": "intermediate"},
    ),
    Document(
        page_content="Implied volatility derived from options prices predicts near-term market swings.",
        metadata={"id": 5, "topic": "derivatives", "year": 2022, "difficulty": "advanced"},
    ),
    Document(
        page_content="Factor investing tilts portfolios toward systematic risk premia like value and momentum.",
        metadata={"id": 6, "topic": "portfolio", "year": 2023, "difficulty": "advanced"},
    ),
]

DIM = 16  # small synthetic dimension — enough for offline demos
rng = np.random.default_rng(seed=0)
EMBEDDINGS = rng.random((len(DOCS), DIM)).astype(np.float32)

print(f"Corpus  : {len(DOCS)} documents")
print(f"Embedding shape: {EMBEDDINGS.shape}  (n_docs x dim)")
print()
for doc in DOCS:
    m = doc.metadata
    print(f"  [id={m['id']}] [{m['topic']:12s}] [{m['year']}] [{m['difficulty']:12s}] {doc.page_content[:55]}...")

### Build the shared FAISS vector store

`FakeEmbeddings` is used here so this notebook runs without any API key.
At query time, FAISS re-encodes the query string using the same model;
with `FakeEmbeddings` the query vector is random, so the ranking is
arbitrary — replace with `OpenAIEmbeddings()` for real semantic results.

In [ ]:
query_encoder = FakeEmbeddings(size=DIM)

vector_store = create_vectorstore(
    docs=DOCS,
    embeddings=EMBEDDINGS,
    fake_embeddings_model=query_encoder,
)

print(f"Vector store ready. Indexed vectors: {vector_store.index.ntotal}")

---
## 3 — `create_retriever`

Wraps `VectorStore.as_retriever()` with a consistent signature.
The `search_type` string controls the retrieval algorithm;
`search_kwargs` passes algorithm-specific parameters.

| `search_type` | Key `search_kwargs` | Use when |
|---|---|---|
| `"similarity"` | `k` | Default — return the top-*k* most similar docs |
| `"mmr"` | `k`, `fetch_k`, `lambda_mult` | You want diversity: similar but not repetitive results |
| `"similarity_score_threshold"` | `score_threshold`, `k` | You only want results above a minimum similarity score |

The returned `VectorStoreRetriever` is callable via `.invoke(query)` like any
other LangChain retriever.

### `search_type="similarity"` — top-*k* results

In [ ]:
similarity_retriever = create_retriever(
    vs=vector_store,
    search_type="similarity",
    search_kwargs={"k": 3},
)

results = similarity_retriever.invoke("portfolio risk management")

print(f"search_type='similarity', k=3  ->  {len(results)} results")
for i, doc in enumerate(results, 1):
    print(f"  {i}. [id={doc.metadata['id']}] {doc.page_content}")

### `search_type="mmr"` — Maximal Marginal Relevance

MMR balances relevance and diversity. It fetches `fetch_k` candidates,
then iteratively selects `k` documents that are relevant to the query
but dissimilar to each other.

- `fetch_k` — candidate pool size (must be >= `k`); larger means more diversity options
- `lambda_mult` — `1.0` = pure relevance, `0.0` = pure diversity; `0.5` is a good default

In [ ]:
mmr_retriever = create_retriever(
    vs=vector_store,
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": 0.5},
)

results_mmr = mmr_retriever.invoke("portfolio risk management")

print(f"search_type='mmr', k=3, fetch_k=6  ->  {len(results_mmr)} results")
for i, doc in enumerate(results_mmr, 1):
    m = doc.metadata
    print(f"  {i}. [id={m['id']}] [{m['topic']}] {doc.page_content}")

### `search_type="similarity_score_threshold"` — score-gated results

Only returns documents whose similarity score is at or above `score_threshold`
(scale depends on the embedding model and distance metric — cosine typically 0–1).
The result set may be **smaller than `k`** if few documents clear the threshold,
and may be **empty**.

> With `FakeEmbeddings` the scores are random, so expect varied output.

In [ ]:
threshold_retriever = create_retriever(
    vs=vector_store,
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.5, "k": 6},
)

results_thresh = threshold_retriever.invoke("portfolio risk management")

print(f"score_threshold=0.5  ->  {len(results_thresh)} result(s) cleared the threshold")
for i, doc in enumerate(results_thresh, 1):
    print(f"  {i}. [id={doc.metadata['id']}] {doc.page_content}")

if not results_thresh:
    print("  (empty — FakeEmbeddings scores are random; lower the threshold or use real embeddings)")

---
## 4 — `create_BM25retriever_from_docs`

Creates a BM25 retriever from a list of `Document` objects.
BM25 is a sparse, keyword-based algorithm — **no embeddings required**.

This function is the canonical definition from `kitai.index`;
`kitai.retriever` re-exports it so callers can import everything from one place.

**When to prefer BM25 over FAISS:**
- Exact term matches matter (product codes, names, identifiers).
- You don't have an embeddings model available.
- You want a fast, cheap baseline to compare against semantic search.

**`k` parameter:** the number of top documents to return per query.
It can be changed after construction via `retriever.k = new_value`.

In [ ]:
bm25_from_docs = create_BM25retriever_from_docs(docs=DOCS, k=3)
print(f"BM25 (from docs) ready.  k={bm25_from_docs.k}  corpus={len(bm25_from_docs.docs)} docs")

query = "drawdown and risk"
bm25_results = bm25_from_docs.invoke(query)

print(f"\nQuery   : {query!r}")
print(f"Results : {len(bm25_results)}")
for i, doc in enumerate(bm25_results, 1):
    print(f"  {i}. [id={doc.metadata['id']}] {doc.page_content}")

### Adjusting `k` after construction

Setting `retriever.k` is the standard LangChain pattern — no need to
rebuild the index when you only want to change the result count.

In [ ]:
bm25_from_docs.k = 2
results_k2 = bm25_from_docs.invoke(query)
print(f"After k=2: {len(results_k2)} result(s)")

bm25_from_docs.k = 3  # reset for remaining sections

---
## 5 — `create_BM25retriever_from_text`

Same as `create_BM25retriever_from_docs` but accepts plain `list[str]`
instead of `list[Document]`. Useful when you have raw strings and don't
need LangChain `Document` metadata.

Internally `BM25Retriever.from_texts` wraps each string in a `Document`
before building the index. The returned retriever is identical to
one built from documents.

In [ ]:
TEXTS = [doc.page_content for doc in DOCS]

bm25_from_text = create_BM25retriever_from_text(docs=TEXTS, k=3)
print(f"BM25 (from text) ready.  k={bm25_from_text.k}  corpus={len(bm25_from_text.docs)} docs")

query = "volatility and options"
text_results = bm25_from_text.invoke(query)

print(f"\nQuery   : {query!r}")
print(f"Results : {len(text_results)}")
for i, doc in enumerate(text_results, 1):
    print(f"  {i}. {doc.page_content}")

---
## 6 — `reorder_docs` — LongContextReorder

Reorders a list of retrieved documents so the most relevant ones appear
at the **beginning and end** of the list, with less relevant ones in
the middle. This matters because LLMs tend to under-weight information
in the middle of long contexts — the "lost-in-the-middle" problem.

```
Input  : [doc1, doc2, doc3, doc4, doc5, doc6]    <- ranked by retriever
Output : [doc6, doc4, doc2, doc1, doc3, doc5]    <- alternating from ends
```

The most relevant document (index 0 from the retriever) is placed **last**;
the second most relevant is placed **first** — both positions where LLM
attention is strongest.

**When to use:** apply `reorder_docs` immediately before building the
context string that is passed to the LLM. Do not use it before a
re-ranking step (re-rankers expect relevance-ordered input).

In [ ]:
# Retrieve 6 docs so the reordering pattern is clearly visible
all_retriever = create_retriever(
    vs=vector_store,
    search_type="similarity",
    search_kwargs={"k": 6},
)
retrieved = all_retriever.invoke("portfolio risk management")
reordered  = reorder_docs(retrieved)

retrieved_ids = [d.metadata["id"] for d in retrieved]
reordered_ids = [d.metadata["id"] for d in reordered]

print(f"Retrieved order  : {retrieved_ids}")
print(f"Reordered order  : {reordered_ids}")
print()
print("Reordered documents (best LLM-visible positions highlighted):")
for i, doc in enumerate(reordered):
    note = "  <- strong LLM attention" if i in (0, len(reordered) - 1) else ""
    print(f"  pos {i}: [id={doc.metadata['id']}] {doc.page_content[:60]}...{note}")

### Building the context string after reordering

In [ ]:
context = "\n\n".join(doc.page_content for doc in reordered)
print("Context to pass to LLM:")
print(context)

---
## 7 — `create_hybrid_retriever` — EnsembleRetriever

Combines a sparse (BM25) retriever and a semantic (vector-store) retriever
using Reciprocal Rank Fusion (RRF). Each retriever produces its own ranked
list; the ensemble merges them by a weighted sum of reciprocal ranks.

```
BM25Retriever    -----+
                      +-> EnsembleRetriever (RRF)  ->  merged list
VectorStoreRetriever -+
```

**`weights_sparse` parameter** controls the BM25 contribution:

| `weights_sparse` | Effect |
|---|---|
| `0.0` | Pure semantic — BM25 results ignored |
| `0.5` | Equal weight (good default) |
| `1.0` | Pure BM25 — semantic results ignored |

The semantic retriever automatically receives weight `1 - weights_sparse`.

**Invariant:** `weights_sparse` must be in `[0, 1]`; anything outside raises `ValueError` immediately.

In [ ]:
# Component retrievers
bm25_retriever = create_BM25retriever_from_docs(docs=DOCS, k=3)
vector_retriever = create_retriever(
    vs=vector_store,
    search_type="similarity",
    search_kwargs={"k": 3},
)

hybrid_retriever = create_hybrid_retriever(
    sparse_retriever=bm25_retriever,
    semantic_retriever=vector_retriever,
    weights_sparse=0.5,
)

print(f"Ensemble weights: {hybrid_retriever.weights}  (sparse, semantic)")

query = "risk management and drawdown"
hybrid_results = hybrid_retriever.invoke(query)

print(f"\nQuery   : {query!r}")
print(f"Results : {len(hybrid_results)}")
for i, doc in enumerate(hybrid_results, 1):
    m = doc.metadata
    print(f"  {i}. [id={m['id']}] [{m['topic']}] {doc.page_content}")

### Comparing BM25 vs semantic vs hybrid

The three retrievers often return overlapping but distinct result sets.
This is the core value of hybrid retrieval: each source compensates for
the other's blind spots.

In [ ]:
query = "portfolio factor investing"

ids_bm25     = [d.metadata["id"] for d in bm25_retriever.invoke(query)]
ids_semantic = [d.metadata["id"] for d in vector_retriever.invoke(query)]
ids_hybrid   = [d.metadata["id"] for d in hybrid_retriever.invoke(query)]

print(f"Query        : {query!r}")
print(f"BM25    ids  : {ids_bm25}")
print(f"Semantic ids : {ids_semantic}")
print(f"Hybrid ids   : {ids_hybrid}")

### Effect of `weights_sparse`

The weight shift is most noticeable when BM25 and semantic retrievers
disagree on the top documents.

In [ ]:
query = "implied volatility derivatives"

for w in [0.0, 0.3, 0.5, 0.7, 1.0]:
    h = create_hybrid_retriever(
        sparse_retriever=bm25_retriever,
        semantic_retriever=vector_retriever,
        weights_sparse=w,
    )
    ids = [d.metadata["id"] for d in h.invoke(query)]
    print(f"  weights_sparse={w:.1f}  ->  {ids}")

---
## 8 — `create_self_query_retriever`

Uses an LLM to parse natural-language queries into a **semantic search + metadata filter**
pair, then runs them against the vector store together. This allows
queries like *"beginner articles about risk from 2020"* to be handled
precisely — the LLM extracts `difficulty=beginner`, `topic=risk`, `year=2020`
as structured filters, and the remaining text becomes the embedding query.

```
"beginner articles about risk from 2020"
         |
         v  LLM  (SelfQueryRetriever)
 semantic query : "risk articles"
 metadata filter: difficulty == "beginner" AND year == 2020
         |
         v
 VectorStore.similarity_search(
     query="risk articles",
     filter={"difficulty": "beginner", "year": 2020}
 )
```

**`metadata_field_info`** describes each filterable field to the LLM.
Use `AttributeInfo` with a clear `description` — the LLM reads it verbatim.

**`verbose=False` (default):** set to `True` to log the structured query
the LLM generated — useful for debugging unexpected filter behaviour.

### Vector store compatibility

`SelfQueryRetriever` requires a vector store with a built-in filter translator.
In `langchain_classic`, the supported stores are:

`chroma`, `pinecone`, `weaviate`, `qdrant`, `milvus`, `elasticsearch`,
`mongodb_atlas`, `pgvector`, `redis`, `opensearch`, and others.

**FAISS does not have a built-in translator in this version.**
The guide uses FAISS for all other sections because it requires no infrastructure.
For self-query, swap in a supported vector store — Chroma is the easiest
local option (`pip install chromadb langchain-chroma`).

> **Requires OPENAI_API_KEY and a compatible vector store.** Cells below are guarded.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

_has_api_key = bool(os.environ.get("OPENAI_API_KEY"))
_has_chroma  = importlib.util.find_spec("chromadb") is not None

if not _has_api_key:
    print("OPENAI_API_KEY not set — skipping self-query demo.")
elif not _has_chroma:
    print("chromadb not installed — skipping self-query demo.")
    print("Install with: pip install chromadb langchain-chroma")
else:
    import importlib.util
    from langchain_chroma import Chroma
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from langchain_classic.chains.query_constructor.schema import AttributeInfo

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # Chroma requires a real embedding model (no FakeEmbeddings).
    embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

    # Build a Chroma vector store from the same corpus
    chroma_store = Chroma.from_documents(
        documents=DOCS,
        embedding=embeddings_model,
    )

    # Describe every metadata field the LLM is allowed to filter on.
    # The 'description' string is read by the LLM — be precise.
    metadata_field_info = [
        AttributeInfo(
            name="topic",
            description="Subject area of the article. One of: 'equities', 'portfolio', 'risk', 'derivatives'.",
            type="string",
        ),
        AttributeInfo(
            name="year",
            description="Publication year of the article (integer between 2020 and 2023).",
            type="integer",
        ),
        AttributeInfo(
            name="difficulty",
            description="Difficulty level. One of: 'beginner', 'intermediate', 'advanced'.",
            type="string",
        ),
    ]

    self_query_retriever = create_self_query_retriever(
        model=model,
        vector_store=chroma_store,
        document_content_description=(
            "Short finance articles covering equities, portfolio theory, risk, and derivatives."
        ),
        metadata_field_info=metadata_field_info,
        verbose=False,
    )

    print("SelfQueryRetriever ready (backed by Chroma).")

### Example queries with implicit metadata filters

The LLM extracts filter conditions from the natural language;
only documents that pass the filter are candidates for the semantic search.

In [ ]:
if not (_has_api_key and _has_chroma):
    print("Skipping — requires OPENAI_API_KEY and chromadb.")
else:
    test_queries = [
        "beginner articles about risk",
        "advanced portfolio articles from 2023",
        "derivatives content",
    ]

    for q in test_queries:
        print(f"Query    : {q!r}")
        results = self_query_retriever.invoke(q)
        if results:
            for doc in results:
                m = doc.metadata
                print(f"  [id={m['id']}] [{m['topic']}] [{m['year']}] [{m['difficulty']}] {doc.page_content[:60]}...")
        else:
            print("  (no results — try relaxing the filter or raising k)")
        print()

### Debugging with `verbose=True`

When a self-query retriever returns unexpected results, enable `verbose=True`
to log the structured query the LLM produced. This shows the exact filter
expression and semantic query string before they hit the vector store.

In [ ]:
if not (_has_api_key and _has_chroma):
    print("Skipping — requires OPENAI_API_KEY and chromadb.")
else:
    verbose_retriever = create_self_query_retriever(
        model=model,
        vector_store=chroma_store,
        document_content_description=(
            "Short finance articles covering equities, portfolio theory, risk, and derivatives."
        ),
        metadata_field_info=metadata_field_info,
        verbose=True,   # structured query is printed above the results
    )

    # The LLM's structured query output appears in the logs before results
    verbose_retriever.invoke("intermediate risk articles from 2020")

---
## 9 — End-to-end pattern

A production RAG pipeline typically combines **hybrid retrieval** with
**LongContextReorder** before passing the context to the LLM.

```
user query
    |
    +-> BM25Retriever          (keyword match)
    +-> VectorStoreRetriever   (semantic match)
              |
              v
        EnsembleRetriever  (RRF merge)
              |
         reorder_docs()    (LongContextReorder)
              |
         context string
              |
             LLM
```

The helper below encapsulates this pattern as a reusable function.
It runs without an API key — the LLM synthesis step is shown as a comment.

In [ ]:
def retrieve_and_reorder(
    query: str,
    *,
    bm25_k: int = 4,
    semantic_k: int = 4,
    weights_sparse: float = 0.5,
    docs: list[Document],
    embeddings: np.ndarray,
    query_encoder,
) -> list[Document]:
    """
    Hybrid retrieve + reorder in one call.

    Returns documents ordered for optimal LLM attention:
    most relevant at both ends, least relevant in the middle.
    """
    vs = create_vectorstore(
        docs=docs, embeddings=embeddings, fake_embeddings_model=query_encoder
    )
    bm25   = create_BM25retriever_from_docs(docs=docs, k=bm25_k)
    vector = create_retriever(vs=vs, search_type="similarity", search_kwargs={"k": semantic_k})
    hybrid = create_hybrid_retriever(
        sparse_retriever=bm25,
        semantic_retriever=vector,
        weights_sparse=weights_sparse,
    )
    return reorder_docs(hybrid.invoke(query))


# Run the pipeline
query = "risk management frameworks and drawdown"

final_docs = retrieve_and_reorder(
    query,
    bm25_k=3,
    semantic_k=3,
    weights_sparse=0.5,
    docs=DOCS,
    embeddings=EMBEDDINGS,
    query_encoder=query_encoder,
)

context = "\n\n".join(doc.page_content for doc in final_docs)

print(f"Query   : {query!r}")
print(f"Context ({len(final_docs)} docs, LLM-ordered):")
print(context)
print()
print("# Pass 'context' to your LLM:")
print("# response = llm.invoke(f'Answer using this context:\\n{context}\\n\\nQuestion: {query}')")

---
## 10 — Error handling reference

Every function validates inputs **before** any expensive operation
(index build, network call). All errors are raised with descriptive messages;
no function returns `None` or `[]` as a failure sentinel.

| Function | Guard condition | Error type |
|---|---|---|
| `create_BM25retriever_from_docs` | `docs` is empty | `ValueError` |
| `create_BM25retriever_from_docs` | `k <= 0` | `ValueError` |
| `create_BM25retriever_from_text` | `docs` is empty | `ValueError` |
| `create_BM25retriever_from_text` | `k <= 0` | `ValueError` |
| `create_hybrid_retriever` | `weights_sparse` not in `[0, 1]` | `ValueError` |
| `create_vectorstore` (index) | `len(docs) != embeddings.shape[0]` | `ValueError` |
| Network / LLM errors | propagated from LangChain | varies |

In [ ]:
# create_BM25retriever_from_text — empty list and non-positive k
try:
    create_BM25retriever_from_text([], k=3)
except ValueError as e:
    print(f"Empty docs list:\n  {type(e).__name__}: {e}\n")

for k_val in [0, -5]:
    try:
        create_BM25retriever_from_text(TEXTS, k=k_val)
    except ValueError as e:
        print(f"k={k_val}:\n  {type(e).__name__}: {e}")
print()

In [ ]:
# create_BM25retriever_from_docs — same guards
try:
    create_BM25retriever_from_docs([], k=3)
except ValueError as e:
    print(f"Empty docs list:\n  {type(e).__name__}: {e}\n")

try:
    create_BM25retriever_from_docs(DOCS, k=0)
except ValueError as e:
    print(f"k=0:\n  {type(e).__name__}: {e}\n")

In [ ]:
# create_hybrid_retriever — weights out of range
for w in [-0.1, 1.5]:
    try:
        create_hybrid_retriever(
            sparse_retriever=bm25_retriever,
            semantic_retriever=vector_retriever,
            weights_sparse=w,
        )
    except ValueError as e:
        print(f"weights_sparse={w}:\n  {type(e).__name__}: {e}")
print()

# Boundary values 0.0 and 1.0 are valid
for w in [0.0, 1.0]:
    h = create_hybrid_retriever(
        sparse_retriever=bm25_retriever,
        semantic_retriever=vector_retriever,
        weights_sparse=w,
    )
    print(f"weights_sparse={w} is valid  ->  weights={h.weights}")